<!--
  id: hitl-healthcare-demo
  type: example
  day: 2_1
  competency: 1.5.1
  use_case: healthcare
-->

# Example 2.1 — Human-in-the-Loop

A clinical intake agent proposes scheduling a follow-up appointment. Before that action actually runs, a human has to review it. This notebook builds the pause-and-resume pattern end to end.

Companion to [Problem Solution Ladder: Human-in-the-Loop](problem-solution-ladder-1.5.1.qmd).

## Setup

In [ ]:
%pip install -q -U langchain langchain-google-genai langgraph

In [ ]:
import os
from google.colab import userdata

os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
print("API key loaded!")

## Step 1 — Build the tool that needs review

In [ ]:
from langchain.tools import tool

@tool
def schedule_followup(patient_id: str, specialist: str, urgency: str) -> str:
    """Schedule a follow-up appointment with a specialist for a patient."""
    return f"Scheduled: patient {patient_id} with {specialist} ({urgency} priority)"

## Step 2 — Wire up the approval gate

`HumanInTheLoopMiddleware` intercepts the call before it runs. `InMemorySaver` is what lets execution actually pause and be resumed later, not just block.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model="google_genai:gemini-2.5-flash",
    tools=[schedule_followup],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "schedule_followup": {"allowed_decisions": ["approve", "edit", "reject"]},
            },
            description_prefix="Clinical action pending review",
        ),
    ],
    checkpointer=InMemorySaver(),
)

## Step 3 — Watch it pause

`version="v2"` is what makes the paused actions available on `result.interrupts`.

In [ ]:
config = {"configurable": {"thread_id": "patient-482-visit"}}

result = agent.invoke(
    {"messages": [{"role": "user", "content": "Patient 482 reports chest pain, schedule an urgent cardiology follow-up."}]},
    config=config,
    version="v2",
)
for action in result.interrupts[0].value["action_requests"]:
    print(action["description"])  # nothing has actually been scheduled yet

## Step 4 — Resolve it as a human reviewer

Decisions come back as a list — one per action under review, in the same order they appeared.

Try `approve` first. Then re-run Step 3 with a new `thread_id` and swap in `{"type": "reject", "message": "..."}` instead — read what comes back to the agent.

In [ ]:
from langgraph.types import Command

final = agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config,
    version="v2",
)
print(final.value["messages"][-1].content)

## Try it yourself

Add a second tool (e.g. a `message_care_team` tool) with a different `allowed_decisions` list than `schedule_followup`, and justify the difference.